# Zonal statistics: a grid averaged over polygons

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/holoviz/lumen/blob/main/examples/notebooks/zonal_statistics.ipynb)

Ask for a gridded variable averaged over regions, and Lumen joins the grid to the polygons in one SQL query: it moves the grid's 0 to 360 longitudes into -180 to 180, weights each cell by the share of it inside a region and by latitude, and keeps the regions smaller than a grid cell. The first part runs offline on sample data; the second reads a slice of ERA5 from Google Cloud, with no account.

In [ ]:
%pip install -q "lumen[ai-openai,xarray,geo]" "hvsampledata>=0.1.5"

In [ ]:
import hvsampledata
import lumen

from lumen.ai.llm import Kilo

llm = Kilo()  # free models, no account; with an API key set, leave out llm= and Lumen uses that provider

## Air temperature by US state

The NCEP air temperature grid (2.5 degrees, longitudes 200 to 330) and the US states go into one source, so a question can join them.

In [ ]:
air = hvsampledata.air_temperature("xarray")
states = hvsampledata.us_states("geopandas")

source = lumen.read({"air": air, "states": states})
answer = source.ask("What is the area-weighted mean air temperature in degrees Celsius in each US state?", llm=llm)
answer

In [ ]:
print(answer.sql)

The query averages each cell over time, then weights it by the area it shares with each state (`ST_Intersection`) and by `COS(RADIANS(lat))`. Testing only cell centres would leave out the eight states smaller than a 2.5 degree cell, such as Rhode Island and Delaware.

Check the answer against geopandas, computed without Lumen:

In [ ]:
import numpy as np
import geopandas as gpd
import shapely

cells = (air.air.mean("time") - 273.15).to_dataframe().reset_index()
cells["lon"] = np.where(cells.lon > 180, cells.lon - 360, cells.lon)
cells = gpd.GeoDataFrame(cells, geometry=shapely.box(cells.lon - 1.25, cells.lat - 1.25, cells.lon + 1.25, cells.lat + 1.25), crs=states.crs)
pieces = gpd.overlay(cells, states[["state", "geometry"]].assign(geometry=states.geometry.make_valid()), keep_geom_type=True)
pieces["w"] = pieces.geometry.area * np.cos(np.deg2rad(pieces.lat))
reference = (pieces.air * pieces.w).groupby(pieces.state).sum() / pieces.w.groupby(pieces.state).sum()

result = answer.data.set_index(answer.data.select_dtypes(exclude="number").columns[0])
value = result.select_dtypes("number").columns[0]
print(len(result), "states; largest difference from geopandas:", round(float((result[value] - reference).abs().max()), 3), "°C")

In [ ]:
chat = source.chat(llm=llm)
chat.ask("What is the area-weighted mean air temperature in degrees Celsius in each US state?")
chat.ask("Map it, coloured by temperature")

## ERA5: three questions

ERA5 is ECMWF's global reanalysis; Google's ARCO-ERA5 copy is public Zarr. Open it with xarray and cut a slice before Lumen sees it: one day at four times over the United States (about 0.4 MB), still on 0 to 360 longitudes.

In [ ]:
import xarray as xr

era5 = xr.open_zarr(
    "https://storage.googleapis.com/gcp-public-data-arco-era5/ar/1959-2022-full_37-1h-0p25deg-chunk-1.zarr-v2", chunks={}
)
t2m = era5[["2m_temperature"]].rename({"2m_temperature": "t2m"}).sel(
    time=["2021-07-01T00", "2021-07-01T06", "2021-07-01T12", "2021-07-01T18"],
    latitude=slice(50, 24), longitude=slice(235, 294),
).load()

era5_states = lumen.read({"t2m": t2m, "states": states})

In [ ]:
hottest = era5_states.ask("Which US state had the highest area-weighted mean 2 m temperature on 1 July 2021, in degrees Celsius?", llm=llm)
hottest

In [ ]:
swing = era5_states.ask("Which US state had the largest difference between its warmest and coldest area-weighted mean 2 m temperature across the four times?", llm=llm)
swing

In [ ]:
mapped = era5_states.ask("Map the area-weighted mean 2 m temperature on 1 July 2021 by US state, in degrees Celsius", llm=llm)
mapped

## Limits

- The polygons must be in longitude and latitude (EPSG:4326); reproject others with `gdf.to_crs(4326)` first.
- Cell areas are measured in degrees on the grid itself, which is what the latitude weight corrects; for large regions near the poles prefer an equal-area grid.
- Remote data is read once into memory here; slice it to what the questions need.